## 16. Summary & Usage Notes

### Entropy-Based Domain Adaptation Implementation

**What was done:**
- Computed Shannon entropy statistics from VinDr-Mammo training set (mean, std)
- Cached statistics for reuse (`cache/entropy_stats_vindr_train.json`)
- Applied iterative entropy transformation to INbreast images:
  - Square image if entropy too low (increases contrast)
  - Square root image if entropy too high (decreases contrast)
  - Iterate until entropy reaches target range [mean-std, mean+std]
- Evaluated adapted model on INbreast without any retraining

**Key Files:**
- `src/domain_adaptation.py` - Entropy calculation and adaptive transform
- `src/adaptive_preprocessing.py` - Preprocessing wrapper with adaptation
- `src/cache_manager.py` - Entropy statistics caching
- `src/datasets.py` - Factory functions for adapted datasets
- `scripts/compute_entropy_stats.py` - One-time entropy computation
- `scripts/evaluate_zeroshot_adapted.py` - Multi-scenario evaluation

**To Run Standalone Scripts:**

```bash
cd /content/drive/MyDrive/breast_cancer_detection

# Compute entropy statistics (one-time, ~1-2 hours)
python scripts/compute_entropy_stats.py

# Evaluate zero-shot with adaptation
python scripts/evaluate_zeroshot_adapted.py \
    --model_path checkpoints/trained_model.pth \
    --dropout 0.2 \
    --unfreeze_frac 0.5 \
    --entropy_cache cache/entropy_stats_vindr_train.json
```

**Expected Improvements:**
- INbreast prediction spread: >50% of [0,1] range (vs baseline ~5-10%)
- AUROC improvement: +0.10 to +0.20 (from 0.54 to 0.64-0.74)
- Better calibration: Lower Brier score

**Next Steps:**
- Run full-scale NSGA-III optimization with adaptation
- Evaluate all Pareto solutions on adapted INbreast
- Compare domain shift mitigation across different hyperparameter configurations

In [ ]:
# Visualize baseline vs adapted performance
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ROC Curves
fpr_baseline, tpr_baseline, _ = roc_curve(y_true_inbreast, y_probs_inbreast)
fpr_adapted, tpr_adapted, _ = roc_curve(y_true_adapted, y_probs_adapted)

axes[0].plot(fpr_baseline, tpr_baseline, 'r--', linewidth=2, alpha=0.7,
             label=f'Baseline (AUC={inbreast_metrics["auroc"]:.3f})')
axes[0].plot(fpr_adapted, tpr_adapted, 'b-', linewidth=2, 
             label=f'Adapted (AUC={adapted_metrics["auroc"]:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.3, label='Random')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves: INbreast Zero-Shot', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Add annotation showing improvement
if auroc_change > 0:
    axes[0].text(0.6, 0.2, f'Δ AUROC: +{auroc_change:.4f}', 
                bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8),
                fontsize=11)

# Precision-Recall Curves
precision_baseline, recall_baseline, _ = precision_recall_curve(y_true_inbreast, y_probs_inbreast)
precision_adapted, recall_adapted, _ = precision_recall_curve(y_true_adapted, y_probs_adapted)

axes[1].plot(recall_baseline, precision_baseline, 'r--', linewidth=2, alpha=0.7,
             label=f'Baseline (AP={inbreast_metrics["pr_auc"]:.3f})')
axes[1].plot(recall_adapted, precision_adapted, 'b-', linewidth=2, 
             label=f'Adapted (AP={adapted_metrics["pr_auc"]:.3f})')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curves: INbreast Zero-Shot', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

# Add annotation showing improvement
if pr_auc_change > 0:
    axes[1].text(0.6, 0.2, f'Δ PR-AUC: +{pr_auc_change:.4f}', 
                bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8),
                fontsize=11)

plt.tight_layout()
plt.show()

# Probability distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram of predictions
axes[0].hist(y_probs_inbreast, bins=30, alpha=0.6, color='red', label='Baseline', edgecolor='black')
axes[0].hist(y_probs_adapted, bins=30, alpha=0.6, color='blue', label='Adapted', edgecolor='black')
axes[0].axvline(optimal_threshold, color='green', linestyle='--', linewidth=2, label=f'Threshold={optimal_threshold:.3f}')
axes[0].set_xlabel('Predicted Probability', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Prediction Distribution on INbreast', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot comparison
axes[1].boxplot([y_probs_inbreast, y_probs_adapted], 
                labels=['Baseline', 'Adapted'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_ylabel('Predicted Probability', fontsize=12)
axes[1].set_title('Prediction Spread Comparison', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Add spread annotations
axes[1].text(1, y_probs_inbreast.min() - 0.05, f'Spread: {baseline_spread:.3f}', 
            ha='center', fontsize=10)
axes[1].text(2, y_probs_adapted.min() - 0.05, f'Spread: {adapted_spread:.3f}', 
            ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\n📊 Visual Analysis Complete!")

In [ ]:
from src.datasets import create_inbreast_dataset_with_adaptation

print("="*80)
print("ZERO-SHOT WITH ENTROPY ADAPTATION")
print("="*80)

# Create adapted INbreast dataset
print("\n[1/2] Creating INbreast dataset with entropy adaptation...")
entropy_cache_path = cache.get_cache_path("vindr_train")

inbreast_adapted = create_inbreast_dataset_with_adaptation(
    dicom_dir=INBREAST_DICOM_DIR,
    csv_file=INBREAST_CSV,
    entropy_stats_path=str(entropy_cache_path),
    benign_birads=[2, 3],
    malignant_birads=[5, 6]
)

print(f"✓ Adapted INbreast dataset created: {len(inbreast_adapted)} images")

# Evaluate with adaptation
print("\n[2/2] Evaluating adapted model on INbreast...")
y_true_adapted, y_probs_adapted = aggregate_breast_level_predictions(
    model, inbreast_adapted, device
)

# Compute adapted metrics
adapted_metrics = compute_metrics(
    y_true_adapted,
    y_probs_adapted,
    threshold=optimal_threshold  # Same threshold from VinDr-Mammo
)

print("\n" + "="*80)
print("RESULTS COMPARISON: BASELINE vs ADAPTED")
print("="*80)

print(f"\n{'Metric':<20} {'Baseline':<15} {'Adapted':<15} {'Change':<12}")
print("-" * 65)

# Threshold-independent metrics
pr_auc_change = adapted_metrics['pr_auc'] - inbreast_metrics['pr_auc']
auroc_change = adapted_metrics['auroc'] - inbreast_metrics['auroc']
brier_change = adapted_metrics['brier'] - inbreast_metrics['brier']

print(f"{'PR-AUC':<20} {inbreast_metrics['pr_auc']:<15.4f} {adapted_metrics['pr_auc']:<15.4f} {pr_auc_change:+12.4f}")
print(f"{'AUROC':<20} {inbreast_metrics['auroc']:<15.4f} {adapted_metrics['auroc']:<15.4f} {auroc_change:+12.4f}")
print(f"{'Brier Score':<20} {inbreast_metrics['brier']:<15.4f} {adapted_metrics['brier']:<15.4f} {brier_change:+12.4f}")

# Prediction spread analysis
baseline_spread = y_probs_inbreast.max() - y_probs_inbreast.min()
adapted_spread = y_probs_adapted.max() - y_probs_adapted.min()
spread_change = adapted_spread - baseline_spread

print(f"\n{'Probability Range Analysis':<20}")
print("-" * 65)
print(f"{'Baseline Range':<20} [{y_probs_inbreast.min():.4f}, {y_probs_inbreast.max():.4f}]  (spread: {baseline_spread:.4f} = {baseline_spread*100:.1f}%)")
print(f"{'Adapted Range':<20} [{y_probs_adapted.min():.4f}, {y_probs_adapted.max():.4f}]  (spread: {adapted_spread:.4f} = {adapted_spread*100:.1f}%)")
print(f"{'Spread Improvement':<20} {spread_change:+.4f} ({spread_change*100:+.1f}%)")

# Threshold-dependent metrics
print(f"\n{'Threshold-Dependent (t={optimal_threshold:.4f})':<20}")
print("-" * 65)
sens_change = adapted_metrics['sensitivity'] - inbreast_metrics['sensitivity']
spec_change = adapted_metrics['specificity'] - inbreast_metrics['specificity']

print(f"{'Sensitivity':<20} {inbreast_metrics['sensitivity']:<15.4f} {adapted_metrics['sensitivity']:<15.4f} {sens_change:+12.4f}")
print(f"{'Specificity':<20} {inbreast_metrics['specificity']:<15.4f} {adapted_metrics['specificity']:<15.4f} {spec_change:+12.4f}")

print(f"\n{'Confusion Matrix (Adapted)':<20}")
print(f"  TN={adapted_metrics['tn']:<4} FP={adapted_metrics['fp']:<4}")
print(f"  FN={adapted_metrics['fn']:<4} TP={adapted_metrics['tp']:<4}")

# Interpretation
print("\n" + "="*80)
print("INTERPRETATION")
print("="*80)

if auroc_change > 0.10 and adapted_spread > 0.50:
    print("\n✓ EXCELLENT: Significant improvement! Domain adaptation successful.")
    print(f"  - AUROC improved by {auroc_change*100:.1f}%")
    print(f"  - Prediction spread increased to {adapted_spread*100:.1f}% of [0,1] range")
elif auroc_change > 0.05 and adapted_spread > 0.30:
    print("\n✓ GOOD: Moderate improvement. Domain adaptation helped.")
    print(f"  - AUROC improved by {auroc_change*100:.1f}%")
    print(f"  - Prediction spread increased to {adapted_spread*100:.1f}% of [0,1] range")
elif auroc_change > 0.00:
    print("\n⚠ PARTIAL: Minor improvement. May need tuning or alternative methods.")
    print(f"  - AUROC improved by {auroc_change*100:.1f}%")
    print(f"  - Prediction spread: {adapted_spread*100:.1f}% of [0,1] range")
else:
    print("\n✗ UNSUCCESSFUL: No improvement. Consider alternative adaptation strategies.")
    print(f"  - AUROC change: {auroc_change*100:+.1f}%")

print("="*80)

In [ ]:
from src.domain_adaptation import compute_dataset_entropy_stats, EntropyStatistics
from src.cache_manager import EntropyCache

print("="*80)
print("COMPUTING ENTROPY STATISTICS (ONE-TIME)")
print("="*80)

# Check cache
cache = EntropyCache(cache_dir=os.path.join(PROJECT_PATH, "cache"))

if cache.exists("vindr_train"):
    print("\n✓ Loading cached entropy statistics...")
    entropy_stats = cache.load("vindr_train")
    
    print(f"\nEntropy Statistics (from cache):")
    print(f"  Mean:     {entropy_stats.mean:.4f} bits")
    print(f"  Std:      {entropy_stats.std:.4f} bits")
    print(f"  Min:      {entropy_stats.min_entropy:.4f} bits")
    print(f"  Max:      {entropy_stats.max_entropy:.4f} bits")
    print(f"  Samples:  {entropy_stats.n_samples}")
    print(f"  Computed: {entropy_stats.computed_date}")
    print(f"\nTarget range for adaptation: [{entropy_stats.mean - entropy_stats.std:.4f}, {entropy_stats.mean + entropy_stats.std:.4f}] bits")
else:
    print("\n⚠ Cache not found. Computing entropy statistics...")
    print("This will take ~1-2 hours for full VinDr-Mammo training set.\n")
    
    from src.adaptive_preprocessing import AdaptiveMammographyPreprocessor
    
    # Create training-only dataset wrapper (same logic as compute_entropy_stats.py)
    class TrainingOnlyDataset:
        """Wrapper to expose only training samples."""
        def __init__(self, base_dataset, train_indices):
            self.base_dataset = base_dataset
            self.train_indices = list(train_indices)
            self.images_root = base_dataset.images_root
            self.samples = [base_dataset.samples[i] for i in train_indices]
        
        def __len__(self):
            return len(self.train_indices)
        
        def __getitem__(self, idx):
            orig_idx = self.train_indices[idx]
            return self.base_dataset[orig_idx]
    
    train_only_dataset = TrainingOnlyDataset(dataset, train_dataset.indices)
    
    # Create adaptive preprocessor (no adaptation, just for grayscale access)
    adaptive_preprocessor = AdaptiveMammographyPreprocessor(
        apply_adaptation=False
    )
    
    print("Computing entropy statistics on training set...")
    print("Progress will be shown below:\n")
    
    entropy_stats = compute_dataset_entropy_stats(
        dataset=train_only_dataset,
        preprocessor=adaptive_preprocessor,
        save_path=None,
        verbose=True
    )
    
    # Set dataset name
    entropy_stats.dataset_name = "VinDr-Mammo-Train"
    
    # Save to cache
    print(f"\nSaving statistics to cache...")
    cache.save(entropy_stats, "vindr_train")
    
    print(f"\n✓ Computation complete!")
    print(f"\nEntropy Statistics:")
    print(f"  Mean:     {entropy_stats.mean:.4f} bits")
    print(f"  Std:      {entropy_stats.std:.4f} bits")
    print(f"  Min:      {entropy_stats.min_entropy:.4f} bits")
    print(f"  Max:      {entropy_stats.max_entropy:.4f} bits")
    print(f"  Samples:  {entropy_stats.n_samples}")
    print(f"\nTarget range for adaptation: [{entropy_stats.mean - entropy_stats.std:.4f}, {entropy_stats.mean + entropy_stats.std:.4f}] bits")
    print(f"\nCache saved to: {cache.get_cache_path('vindr_train')}")

print("\n" + "="*80)

# Breast Cancer Detection - Google Colab Demo
## Multi-Objective Optimization with NSGA-III

This notebook demonstrates the complete pipeline for breast cancer detection under dataset shift.

**Steps:**
1. Setup environment and install dependencies
2. Mount Google Drive and load datasets
3. Test preprocessing pipeline
4. Test model training
5. Run NSGA-III optimization (small scale)
6. Zero-shot evaluation on INbreast

## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q pydicom opencv-python-headless scikit-image
!pip install -q pymoo
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Import core libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive and Setup Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Upload the breast_cancer_detection folder to your Google Drive
# Then update this path to point to it

PROJECT_PATH = "/content/drive/MyDrive/breast_cancer_detection"

# Verify the path exists
if os.path.exists(PROJECT_PATH):
    print(f"✓ Project found at: {PROJECT_PATH}")
    # Add to Python path
    sys.path.insert(0, PROJECT_PATH)
else:
    print(f"✗ Project not found at: {PROJECT_PATH}")
    print("Please upload the breast_cancer_detection folder to your Google Drive")

In [ ]:
# Configure data paths
# UPDATE THESE PATHS to match your Google Drive structure

VINDR_IMAGES_ROOT = "/content/drive/MyDrive/vindr-mammo/images"
VINDR_CSV = "/content/drive/MyDrive/vindr-mammo/metadata/stratified_selection.csv"

INBREAST_DICOM_DIR = "/content/drive/MyDrive/INbreast/AllDICOMs"
INBREAST_CSV = "/content/drive/MyDrive/INbreast/INbreast.csv"

# Verify paths
print("Checking data paths...")
print(f"VinDr images: {os.path.exists(VINDR_IMAGES_ROOT)}")
print(f"VinDr CSV: {os.path.exists(VINDR_CSV)}")
print(f"INbreast DICOM: {os.path.exists(INBREAST_DICOM_DIR)}")
print(f"INbreast CSV: {os.path.exists(INBREAST_CSV)}")

## 3. Test Preprocessing Pipeline

In [ ]:
from src.preprocessing import MammographyPreprocessor

# Create preprocessor
preprocessor = MammographyPreprocessor()
print("✓ Preprocessor created successfully")

In [ ]:
# Test on a sample DICOM file
# Replace with an actual path from your dataset
sample_dicom = "/content/drive/MyDrive/vindr-mammo/images/STUDY_ID/IMAGE_ID.dicom"

if os.path.exists(sample_dicom):
    processed_img = preprocessor(sample_dicom)
    
    print(f"Processed image shape: {processed_img.shape}")
    print(f"Expected shape: (480, 720, 3)")
    print(f"Value range: [{processed_img.min()}, {processed_img.max()}]")
    
    # Visualize
    plt.figure(figsize=(8, 6))
    plt.imshow(processed_img)
    plt.title("Preprocessed Mammogram")
    plt.axis('off')
    plt.show()
else:
    print(f"Sample file not found. Please update the path.")

## 4. Test Dataset Loading

In [ ]:
from src.datasets import VinDRMammoBinaryDataset, create_breast_level_splits

# Load dataset
print("Loading VinDr-Mammo dataset...")
dataset = VinDRMammoBinaryDataset(
    images_root=VINDR_IMAGES_ROOT,
    csv_file=VINDR_CSV,
    preprocessor=preprocessor
)

print(f"\n✓ Dataset loaded: {len(dataset)} samples")

In [ ]:
# Create train/val split at BREAST LEVEL (preserves CC + MLO pairs)
print("Creating breast-level train/val split...\n")

train_dataset, val_dataset = create_breast_level_splits(
    dataset=dataset,
    train_ratio=0.8,
    random_state=42,
    stratify=True  # Stratify by breast-level labels
)

# Get breast groups for validation set to show statistics
breast_groups = dataset.get_breast_groups()

# Compute class distribution from train subset
train_labels = []
for idx in train_dataset.indices:
    train_labels.append(dataset.samples[idx][-1])

n_benign = sum(1 for l in train_labels if l == 0)
n_malignant = sum(1 for l in train_labels if l == 1)

print(f"\nClass distribution:")
print(f"  Benign: {n_benign}")
print(f"  Malignant: {n_malignant}")
print(f"  Positive weight: {n_benign/n_malignant:.3f}")

In [ ]:
# Test loading a sample
img, label = dataset[0]

print(f"\nSample data:")
print(f"  Image shape: {img.shape}")
print(f"  Image dtype: {img.dtype}")
print(f"  Value range: [{img.min():.3f}, {img.max():.3f}]")
print(f"  Label: {label.item()} ({'Malignant' if label.item() == 1 else 'Benign'})")

# Visualize
plt.figure(figsize=(8, 6))
plt.imshow(img.permute(1, 2, 0))
plt.title(f"Sample Image - {'Malignant' if label.item() == 1 else 'Benign'}")
plt.axis('off')
plt.show()

## 5. Test Model Building

In [ ]:
from src.models import build_resnet152

# Build model with different configurations
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Testing model configurations...\n")

# Test 1: Full fine-tuning
model1 = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=1.0
)
info1 = model1.get_trainable_params_info()
print("Config 1 - Full fine-tuning (unfreeze=1.0):")
print(f"  Total params: {info1['total_params']:,}")
print(f"  Trainable: {info1['trainable_params']:,} ({info1['trainable_percentage']:.1f}%)\n")

# Test 2: Partial fine-tuning
model2 = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=0.5
)
info2 = model2.get_trainable_params_info()
print("Config 2 - Partial fine-tuning (unfreeze=0.5):")
print(f"  Total params: {info2['total_params']:,}")
print(f"  Trainable: {info2['trainable_params']:,} ({info2['trainable_percentage']:.1f}%)\n")

# Test 3: Feature extraction only
model3 = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=0.0
)
info3 = model3.get_trainable_params_info()
print("Config 3 - Feature extraction (unfreeze=0.0):")
print(f"  Total params: {info3['total_params']:,}")
print(f"  Trainable: {info3['trainable_params']:,} ({info3['trainable_percentage']:.1f}%)")

# Test forward pass
model1 = model1.to(device)
test_input = torch.randn(2, 3, 480, 720).to(device)
output = model1(test_input)
print(f"\n✓ Forward pass successful: Input {test_input.shape} → Output {output.shape}")

## 6. Test Augmentation

In [ ]:
from src.augmentations import get_augmentation

# Test different augmentation strengths
img_sample = dataset[0][0]  # Get first image

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

strengths = [0.0, 0.3, 0.6, 1.0]

for i, strength in enumerate(strengths):
    aug = get_augmentation(strength)
    img_aug = aug(img_sample.clone())
    
    axes[i].imshow(img_aug.permute(1, 2, 0))
    axes[i].set_title(f"Augmentation Strength: {strength}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("✓ Augmentation test completed")

## 7. Test Training Pipeline (Quick Demo)

In [ ]:
from src.training import train_model
from torch.utils.data import DataLoader

# Create small subset for quick testing (10% of data, preserving breast groups)
# First create a temporary small dataset with breast-level split
print("Creating small dataset for quick testing...")

# Get 10% of breast groups for quick demo
all_breast_groups = dataset.get_breast_groups()
n_test_breasts = max(5, len(all_breast_groups) // 10)  # At least 5 breasts

# Select random breasts for quick test
import random
random.seed(42)
test_breast_groups = random.sample(all_breast_groups, n_test_breasts)

# Collect all image indices from these breasts
test_indices = []
for group in test_breast_groups:
    test_indices.extend(group["image_indices"])

# Split into train/val (80/20) while preserving breast groups
test_train_indices = []
test_val_indices = []

n_train_breasts = int(len(test_breast_groups) * 0.8)
for i, group in enumerate(test_breast_groups):
    if i < n_train_breasts:
        test_train_indices.extend(group["image_indices"])
    else:
        test_val_indices.extend(group["image_indices"])

small_train_dataset = Subset(dataset, test_train_indices)
small_val_dataset = Subset(dataset, test_val_indices)

print(f"Quick test with {len(small_train_dataset)} train, {len(small_val_dataset)} val samples")
print(f"  Train breasts: {n_train_breasts}")
print(f"  Val breasts: {len(test_breast_groups) - n_train_breasts}")

# Create dataloaders
train_loader = DataLoader(
    small_train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    small_val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Build model
model = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=0.5  # Partial fine-tuning for speed
).to(device)

print("\nStarting quick training test (5 epochs max)...\n")

In [ ]:
# Train for a few epochs
model, metrics = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    val_dataset=small_val_dataset,
    device=device,
    learning_rate=1e-4,
    weight_decay=1e-4,
    patience=3,
    max_epochs=5,  # Just 5 epochs for testing
    pos_weight=n_benign/n_malignant,
    verbose=True
)

print("\n" + "="*80)
print("Training test completed!")
print(f"Final PR-AUC: {metrics['pr_auc']:.4f}")
print(f"Final AUROC: {metrics['auroc']:.4f}")
print(f"Final Brier: {metrics['brier']:.4f}")
print(f"Robustness Degradation: {metrics['robustness_degradation']:.4f}")
print("="*80)

## 8. Test Breast-Level Aggregation (Noisy-OR)

In [ ]:
from src.evaluation import noisy_or_aggregation, aggregate_breast_level_predictions

# Test Noisy-OR formula
print("Testing Noisy-OR aggregation formula:\n")

# Example: CC and MLO views of same breast
test_cases = [
    ([0.2, 0.3], "Low probabilities"),
    ([0.7, 0.8], "High probabilities"),
    ([0.1, 0.9], "Mixed probabilities"),
    ([0.5, 0.5], "Equal probabilities")
]

for probs, description in test_cases:
    result = noisy_or_aggregation(probs)
    print(f"{description}:")
    print(f"  Views: CC={probs[0]:.2f}, MLO={probs[1]:.2f}")
    print(f"  Breast-level: {result:.4f}")
    print(f"  Formula check: 1 - (1-{probs[0]:.2f})*(1-{probs[1]:.2f}) = {result:.4f}\n")

In [ ]:
# Test on real dataset
print("\nTesting breast-level aggregation on validation set...\n")

y_true_breast, y_probs_breast = aggregate_breast_level_predictions(
    model, small_val_dataset, device
)

print(f"Number of breasts evaluated: {len(y_true_breast)}")
print(f"Benign breasts: {sum(y_true_breast == 0)}")
print(f"Malignant breasts: {sum(y_true_breast == 1)}")
print(f"\nBreast-level probability range: [{y_probs_breast.min():.3f}, {y_probs_breast.max():.3f}]")

## 9. Test Robustness Evaluation

In [ ]:
from src.robustness import RobustnessTester

# Test robustness degradation
print("Testing robustness to perturbations...\n")

tester = RobustnessTester(
    brightness_delta=0.1,
    contrast_factor=0.1,
    noise_std=0.02
)

results = tester.evaluate_robustness(model, val_loader, device)

print(f"PR-AUC (clean): {results['pr_auc_standard']:.4f}")
print(f"PR-AUC (perturbed): {results['pr_auc_perturbed']:.4f}")
print(f"Robustness Degradation: {results['degradation']:.4f}")
print(f"\nLower degradation = more robust model")

## 10. NSGA-III Optimization (Small Scale Demo)

**Warning:** Full optimization with 20 population × 50 generations = 1000 evaluations will take 20-50 hours.

This demo runs a **small-scale** version (5 population × 3 generations = 15 evaluations) for demonstration.

In [ ]:
from src.optimization import BreastCancerOptimizationProblem
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.optimize import minimize
from pymoo.util.ref_dirs import get_reference_directions

print("Setting up NSGA-III optimization (DEMO VERSION - small scale)...\n")

# Create problem
problem = BreastCancerOptimizationProblem(
    train_dataset=small_train_dataset,  # Using small subset
    val_dataset=small_val_dataset,
    device=device,
    batch_size=4,
    num_workers=2,
    patience=3,
    max_epochs=5,  # Reduced epochs for demo
    pos_weight=n_benign/n_malignant,
    random_seed=42
)

# Generate reference directions for 4 objectives
ref_dirs = get_reference_directions("das-dennis", 4, n_partitions=3)
print(f"Reference directions: {len(ref_dirs)}")

# Create algorithm
algorithm = NSGA3(
    ref_dirs=ref_dirs,
    pop_size=5  # Small population for demo
)

print("\nStarting optimization...")
print("This will take ~15-30 minutes for 5 pop × 3 gen = 15 evaluations\n")

In [ ]:
# Run optimization
res = minimize(
    problem,
    algorithm,
    termination=("n_gen", 3),  # Just 3 generations for demo
    seed=42,
    verbose=True
)

print("\n" + "="*80)
print("Optimization completed!")
print(f"Pareto solutions found: {len(res.F)}")
print("="*80)

In [ ]:
# Display Pareto front
print("\nPareto Front Solutions:\n")
print(f"{'ID':<5} {'PR-AUC':>8} {'AUROC':>8} {'Brier':>8} {'Robust':>8}")
print("-" * 45)

for i, f in enumerate(res.F):
    pr_auc = -f[0]  # Convert back from minimization
    auroc = -f[1]
    brier = f[2]
    robust = f[3]
    
    print(f"{i:<5} {pr_auc:>8.4f} {auroc:>8.4f} {brier:>8.4f} {robust:>8.4f}")

print("\nNote: This is a DEMO with reduced scale.")
print("For production, use: pop_size=20, n_gen=50 in run_nsga3.py")

## 11. Visualize Pareto Front

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Extract objectives
pr_auc = -res.F[:, 0]
auroc = -res.F[:, 1]
brier = res.F[:, 2]
robust = res.F[:, 3]

# 2D scatter plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# PR-AUC vs AUROC
axes[0, 0].scatter(pr_auc, auroc, c='blue', s=100)
axes[0, 0].set_xlabel('PR-AUC')
axes[0, 0].set_ylabel('AUROC')
axes[0, 0].set_title('PR-AUC vs AUROC')
axes[0, 0].grid(True)

# PR-AUC vs Brier
axes[0, 1].scatter(pr_auc, brier, c='red', s=100)
axes[0, 1].set_xlabel('PR-AUC')
axes[0, 1].set_ylabel('Brier Score')
axes[0, 1].set_title('PR-AUC vs Brier')
axes[0, 1].grid(True)

# PR-AUC vs Robustness
axes[0, 2].scatter(pr_auc, robust, c='green', s=100)
axes[0, 2].set_xlabel('PR-AUC')
axes[0, 2].set_ylabel('Robustness Degradation')
axes[0, 2].set_title('PR-AUC vs Robustness')
axes[0, 2].grid(True)

# AUROC vs Brier
axes[1, 0].scatter(auroc, brier, c='purple', s=100)
axes[1, 0].set_xlabel('AUROC')
axes[1, 0].set_ylabel('Brier Score')
axes[1, 0].set_title('AUROC vs Brier')
axes[1, 0].grid(True)

# AUROC vs Robustness
axes[1, 1].scatter(auroc, robust, c='orange', s=100)
axes[1, 1].set_xlabel('AUROC')
axes[1, 1].set_ylabel('Robustness Degradation')
axes[1, 1].set_title('AUROC vs Robustness')
axes[1, 1].grid(True)

# Brier vs Robustness
axes[1, 2].scatter(brier, robust, c='brown', s=100)
axes[1, 2].set_xlabel('Brier Score')
axes[1, 2].set_ylabel('Robustness Degradation')
axes[1, 2].set_title('Brier vs Robustness')
axes[1, 2].grid(True)

plt.tight_layout()
plt.show()

## 12. Test INbreast Dataset Loading

In [ ]:
from src.datasets import INbreastDataset

# Load INbreast dataset
print("Loading INbreast dataset...\n")

inbreast_dataset = INbreastDataset(
    dicom_dir=INBREAST_DICOM_DIR,
    csv_file=INBREAST_CSV,
    preprocessor=preprocessor
)

print(f"✓ INbreast dataset loaded: {len(inbreast_dataset)} samples")

# Test sample
img, label = inbreast_dataset[0]
print(f"\nSample shape: {img.shape}")
print(f"Sample label: {label.item()}")

# Visualize
plt.figure(figsize=(8, 6))
plt.imshow(img.permute(1, 2, 0))
plt.title(f"INbreast Sample - {'Malignant' if label.item() == 1 else 'Benign'}")
plt.axis('off')
plt.show()

## 13. Summary & Next Steps

### ✓ All Components Tested Successfully

1. **Preprocessing Pipeline** - Converts DICOM to 720×480 RGB with magma colormap
2. **Dataset Loading** - VinDr-Mammo and INbreast datasets working
3. **Model Architecture** - ResNet152 with partial fine-tuning control
4. **Augmentation** - Intensity-based augmentation with strength controller
5. **Training Pipeline** - Early stopping, class balancing, breast-level evaluation
6. **Robustness Testing** - Perturbation-based robustness measurement
7. **NSGA-III Optimization** - Multi-objective hyperparameter optimization

### 🚀 To Run Full-Scale Optimization:

```bash
cd /content/drive/MyDrive/breast_cancer_detection

# Run full NSGA-III (20-50 hours)
python scripts/run_nsga3.py \
    --pop_size 20 \
    --n_gen 50 \
    --max_epochs 100 \
    --run_id "full_run_001"

# Evaluate on INbreast (requires saved model checkpoints)
python scripts/evaluate_zeroshot.py \
    --results_file checkpoints/nsga3_results_full_run_001.pkl \
    --checkpoint_dir checkpoints/ \
    --run_id "full_run_001"
```

### 📊 Expected Runtime:
- Single model training: 10-60 minutes (depends on early stopping)
- Full NSGA-III (1000 evaluations): 20-50 hours on GPU
- Zero-shot evaluation: 1-2 hours for all Pareto solutions

### 📝 Results Will Be Saved In:
- `logs/nsga3_run_*.csv` - Hyperparameters and objectives per evaluation
- `checkpoints/nsga3_results_*.pkl` - Pareto front solutions
- `logs/zeroshot_evaluation_*.csv` - INbreast evaluation results

In [ ]:
# Visualize zero-shot performance comparison
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ROC Curves
fpr_vindr, tpr_vindr, _ = roc_curve(y_true_vindr, y_probs_vindr)
fpr_inbreast, tpr_inbreast, _ = roc_curve(y_true_inbreast, y_probs_inbreast)

axes[0].plot(fpr_vindr, tpr_vindr, 'b-', linewidth=2, 
             label=f'VinDr-Mammo (AUC={vindr_metrics["auroc"]:.3f})')
axes[0].plot(fpr_inbreast, tpr_inbreast, 'r-', linewidth=2, 
             label=f'INbreast Zero-Shot (AUC={inbreast_metrics["auroc"]:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves: Source vs Target Domain', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curves
precision_vindr, recall_vindr, _ = precision_recall_curve(y_true_vindr, y_probs_vindr)
precision_inbreast, recall_inbreast, _ = precision_recall_curve(y_true_inbreast, y_probs_inbreast)

axes[1].plot(recall_vindr, precision_vindr, 'b-', linewidth=2, 
             label=f'VinDr-Mammo (AP={vindr_metrics["pr_auc"]:.3f})')
axes[1].plot(recall_inbreast, precision_inbreast, 'r-', linewidth=2, 
             label=f'INbreast Zero-Shot (AP={inbreast_metrics["pr_auc"]:.3f})')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curves: Source vs Target Domain', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
if inbreast_metrics['auroc'] > 0.85:
    print("✓ Excellent zero-shot generalization (AUROC > 0.85)")
elif inbreast_metrics['auroc'] > 0.75:
    print("✓ Good zero-shot generalization (AUROC > 0.75)")
else:
    print("⚠ Moderate domain shift impact (AUROC < 0.75)")
    
performance_drop = vindr_metrics['auroc'] - inbreast_metrics['auroc']
if performance_drop < 0.05:
    print("✓ Minimal performance degradation (<5% drop)")
elif performance_drop < 0.10:
    print("✓ Small performance degradation (5-10% drop)")
else:
    print(f"⚠ Noticeable performance degradation ({performance_drop*100:.1f}% drop)")

In [ ]:
from src.datasets import INbreastDataset
from src.evaluation import aggregate_breast_level_predictions, compute_metrics
from sklearn.metrics import roc_curve, roc_auc_score, average_precision_score, brier_score_loss

print("="*80)
print("ZERO-SHOT EVALUATION ON INBREAST")
print("="*80)

# Step 1: Load INbreast dataset
print("\n[1/4] Loading INbreast dataset...")
inbreast_dataset = INbreastDataset(
    dicom_dir=INBREAST_DICOM_DIR,
    csv_file=INBREAST_CSV,
    preprocessor=preprocessor,
    benign_birads=[2, 3],     # INbreast uses BI-RADS 2-3 for benign
    malignant_birads=[5, 6]   # BI-RADS 5-6 for malignant
)

print(f"✓ INbreast loaded: {len(inbreast_dataset)} images")

# Get breast groups for INbreast
inbreast_breast_groups = inbreast_dataset.get_breast_groups()
print(f"✓ Total breasts in INbreast: {len(inbreast_breast_groups)}")

# Step 2: Get breast-level predictions on VinDr-Mammo validation (source domain)
print("\n[2/4] Evaluating on VinDr-Mammo validation set (source domain)...")
y_true_vindr, y_probs_vindr = aggregate_breast_level_predictions(
    model, val_dataset, device
)

vindr_metrics = {
    'pr_auc': average_precision_score(y_true_vindr, y_probs_vindr),
    'auroc': roc_auc_score(y_true_vindr, y_probs_vindr),
    'brier': brier_score_loss(y_true_vindr, y_probs_vindr)
}

print(f"VinDr-Mammo Validation Performance:")
print(f"  PR-AUC: {vindr_metrics['pr_auc']:.4f}")
print(f"  AUROC: {vindr_metrics['auroc']:.4f}")
print(f"  Brier: {vindr_metrics['brier']:.4f}")

# Step 3: Determine decision threshold from VinDr-Mammo validation
print("\n[3/4] Determining optimal threshold from VinDr-Mammo validation...")

# Find threshold that maximizes Youden's index (sensitivity + specificity - 1)
fpr, tpr, thresholds = roc_curve(y_true_vindr, y_probs_vindr)
youden_index = tpr - fpr
best_threshold_idx = np.argmax(youden_index)
optimal_threshold = thresholds[best_threshold_idx]

print(f"✓ Optimal threshold: {optimal_threshold:.4f}")
print(f"  At this threshold on VinDr-Mammo validation:")
print(f"    Sensitivity: {tpr[best_threshold_idx]:.4f}")
print(f"    Specificity: {1 - fpr[best_threshold_idx]:.4f}")

# Step 4: Zero-shot evaluation on INbreast (NO retraining, NO threshold tuning)
print("\n[4/4] Zero-shot evaluation on INbreast (target domain)...")
print("  - NO fine-tuning")
print("  - NO threshold adjustment")
print("  - Same preprocessing")
print("  - Same Noisy-OR aggregation\n")

y_true_inbreast, y_probs_inbreast = aggregate_breast_level_predictions(
    model, inbreast_dataset, device
)

# Compute metrics
inbreast_metrics = compute_metrics(
    y_true_inbreast, 
    y_probs_inbreast, 
    threshold=optimal_threshold  # Use threshold from VinDr-Mammo
)

print("\n" + "="*80)
print("ZERO-SHOT RESULTS ON INBREAST")
print("="*80)

print("\nThreshold-Independent Metrics:")
print(f"  PR-AUC:  {inbreast_metrics['pr_auc']:.4f}")
print(f"  AUROC:   {inbreast_metrics['auroc']:.4f}")
print(f"  Brier:   {inbreast_metrics['brier']:.4f}")

print(f"\nThreshold-Dependent Metrics (threshold={optimal_threshold:.4f} from VinDr-Mammo):")
print(f"  Sensitivity: {inbreast_metrics['sensitivity']:.4f}")
print(f"  Specificity: {inbreast_metrics['specificity']:.4f}")
print(f"  Precision:   {inbreast_metrics['precision']:.4f}")
print(f"  Recall:      {inbreast_metrics['recall']:.4f}")

print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {inbreast_metrics['tn']}")
print(f"  False Positives: {inbreast_metrics['fp']}")
print(f"  False Negatives: {inbreast_metrics['fn']}")
print(f"  True Positives:  {inbreast_metrics['tp']}")

# Compare source vs target domain
print("\n" + "="*80)
print("DOMAIN SHIFT ANALYSIS")
print("="*80)
print(f"\n{'Metric':<20} {'VinDr-Mammo':<15} {'INbreast':<15} {'Δ':<10}")
print("-" * 60)
print(f"{'PR-AUC':<20} {vindr_metrics['pr_auc']:<15.4f} {inbreast_metrics['pr_auc']:<15.4f} {inbreast_metrics['pr_auc'] - vindr_metrics['pr_auc']:<10.4f}")
print(f"{'AUROC':<20} {vindr_metrics['auroc']:<15.4f} {inbreast_metrics['auroc']:<15.4f} {inbreast_metrics['auroc'] - vindr_metrics['auroc']:<10.4f}")
print(f"{'Brier Score':<20} {vindr_metrics['brier']:<15.4f} {inbreast_metrics['brier']:<15.4f} {inbreast_metrics['brier'] - vindr_metrics['brier']:<10.4f}")

print("\n✓ Zero-shot evaluation completed!")
print("="*80)

## 14. Zero-Shot Evaluation on INbreast

Transfer the trained model to INbreast **without any fine-tuning or threshold adjustment**.

## 15. Entropy-Based Domain Adaptation

The zero-shot results above may show domain shift. Here we apply **entropy-based domain adaptation** to align INbreast intensity distribution with VinDr-Mammo training statistics.